# CS781 Phase 1 - continual pretraining of ONE model to a common step count
Goal: bring every model to the same total number of optimizer steps (`TARGET_STEPS`, 131,072 tokens/step) so the models are compared at equal tokens.
Attach as inputs: the **code dataset**, the **data dataset**, and the **training output of this model** (the Kaggle model/dataset that holds `run_<name>/final.pt`).
Set `MODEL` (and `MB`), then **Save Version -> Save & Run All** (GPU T4 x2).
* weights are loaded from `final.pt`, the optimizer is fresh, LR warms up to `LR` and decays to 0 over the last 20% of the *extra* steps
* the data stream continues from the position stored in `final.pt` (unseen windows first)
* the run stops by itself at `TARGET_STEPS` total, or earlier if the time limit is reached (LR decay is then squeezed to fit, as before)
Outputs: `/kaggle/working/run_<name>_cpt/` (`final.pt`, `latest.pt`, `train_log.txt`, `metrics.jsonl`, `DONE`). Step counters in the log restart at 0 (extra steps only).

In [ ]:
import os, sys, json, re, glob, shutil, subprocess, time, threading
DATA = CODE_SRC = "/kaggle/input"      # auto-discovered below
WORK = "/kaggle/working/cs781/train"

def find(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(sorted(hits, key=len)[0])
DATA = find(DATA, "manifest.json")
CODE_SRC = find(CODE_SRC, "train.py")
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)
print("data :", DATA, os.listdir(DATA))
print("code :", WORK, sorted(os.listdir(WORK)))
for f in ["sampler.py", "train.py", "generate.py", "tests.py", "configs/e.json"]:
    assert os.path.exists(f), f"MISSING {f} - re-upload the latest train_code.zip as the dataset"
assert "test_sampler" in open("tests.py").read(), "tests.py is an OLD version - re-upload the latest code"
assert "smoke_steps" in open("train.py").read(), "train.py is an OLD version - re-upload the latest code"
os.makedirs("/kaggle/working/logs", exist_ok=True)

def run_parallel(jobs, timeout=None):
    """jobs: list of (tag, shell_cmd, gpu_index_or_None). Runs them at the same time (one per GPU), streams
    output prefixed with the tag, saves /kaggle/working/logs/<tag>.txt, returns {tag: (exit_code, text)}."""
    procs, outs, lock = {}, {}, threading.Lock()
    def reader(tag, p):
        buf = []
        for line in p.stdout:
            buf.append(line)
            with lock:
                print(f"[{tag}] {line}", end="", flush=True)
        outs[tag] = buf
    threads = []
    for tag, cmd, gpu in jobs:
        env = dict(os.environ)
        if gpu is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        print(f"$ [{tag}] (GPU {gpu}) {cmd}", flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env, start_new_session=True)
        procs[tag] = p
        t = threading.Thread(target=reader, args=(tag, p), daemon=True); t.start(); threads.append(t)
    timer = None
    if timeout:
        def kill():
            print("TIMEOUT - killing", flush=True)
            for p in procs.values():
                try: os.killpg(p.pid, 9)
                except Exception: pass
        timer = threading.Timer(timeout, kill); timer.start()
    for p in procs.values(): p.wait()
    for t in threads: t.join()
    if timer: timer.cancel()
    res = {}
    for tag, p in procs.items():
        txt = "".join(outs.get(tag, []))
        open(f"/kaggle/working/logs/{tag}.txt", "w").write(txt)
        res[tag] = (p.returncode, txt)
        print(f"[{tag}] exit {p.returncode}", flush=True)
    return res

In [ ]:
MODEL = "b"                 # a, b, c, d, e
MB = 16                     # per-GPU micro-batch: a, b, c, d = 16 ; e = 32
EXTRA = "--compile"
TARGET_STEPS = 13000        # common total steps for B, C, D (E and A are already above it -> not continued)
LR = 2e-4                   # peak LR of the continuation (original run: 6e-4)
WARMUP = 100
MAX_HOURS = 11.5            # D needs ~11 h; the Kaggle limit is 12 h
RESUME = None               # only to restart THIS continuation after a crash: ".../run_<name>_cpt/latest.pt"

cfg = json.load(open(f"configs/{MODEL}.json")); NAME = cfg["name"]
GB = cfg["train"]["global_batch_tokens"]
hits = [p for p in glob.glob("/kaggle/input/**/final.pt", recursive=True) if f"run_{MODEL}_" in p]
assert hits, f"final.pt of model {MODEL} not found under /kaggle/input - attach the training output of model {MODEL}"
PREV = sorted(hits, key=len)[0]
import torch
ck = torch.load(PREV, map_location="cpu")
prev_steps = ck["state"]["step"]; has_data = bool(ck["state"].get("data"))
del ck
extra_steps = TARGET_STEPS - prev_steps
print(f"previous run : {PREV}")
print(f"steps done   : {prev_steps}   (data position stored: {has_data})")
print(f"extra steps  : {extra_steps}  -> total {TARGET_STEPS}   ({extra_steps * GB / 1e9:.2f}B more tokens)")
assert has_data, "no data position in final.pt - cannot continue the data stream"
assert extra_steps > 0, f"model {MODEL} already has {prev_steps} >= {TARGET_STEPS} steps - nothing to do, do NOT continue this model"
TORCHRUN = "torchrun --standalone --nproc_per_node=2"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

## Continual pretraining run

In [ ]:
cmd = (f"{TORCHRUN} train.py --config configs/{MODEL}.json --data_root {DATA} --out /kaggle/working/run_{NAME}_cpt "
       f"--micro_batch {MB} --max_hours {MAX_HOURS} --init_from {PREV} --data_state {PREV} "
       f"--lr {LR} --warmup {WARMUP} --budget_tokens {extra_steps * GB} {EXTRA}" + (f" --resume {RESUME}" if RESUME else ""))
res = run_parallel([(f"cpt_{MODEL}", cmd, None)])
rc, txt = res[f"cpt_{MODEL}"]
print("exit code", rc, [l for l in txt.splitlines() if "DONE" in l][-1:])
assert "continual pretraining: weights loaded" in txt and "data stream continues" in txt and "NO SAMPLER STATE" not in txt

In [ ]:
try:
    import pandas as pd, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    df = pd.read_json(f"/kaggle/working/run_{NAME}_cpt/metrics.jsonl", lines=True)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    tr = df[df["loss"].notna()]; ax[0].plot(tr["step"], tr["loss"]); ax[0].set_title("train loss")
    ev = df[df["eval"].notna()]; ax[1].plot(ev["step"], [e["avg"] for e in ev["eval"]], marker="o"); ax[1].set_title("val loss (shared metric)")
    plt.savefig("/kaggle/working/curves.png", dpi=110)
except Exception as e:
    print("plot skipped:", e)